# Punto 8 – Static Matplotlib Charts from Amazon Athena SQL (US 5.1)
Executes the 5 Athena descriptive SQL queries from `SQLScripts/athena/` against `stocks_db.stocks_enriched`, loads the CSV results from S3 via `boto3`, and exports 5 static Matplotlib `.png` charts into `punto8/`.

In [ ]:
%pip install -q boto3 pandas matplotlib

In [3]:
import getpass
import io
import os
from pathlib import Path
import time
import urllib.request

import boto3
import matplotlib.pyplot as plt
import pandas as pd

BUCKET = os.environ.get("S3_BUCKET", "stock-big-data-bucket")
REGION = os.environ.get("AWS_REGION", "us-east-1")
RAW_SQL_BASE = "https://raw.githubusercontent.com/DavidCurelop/BigDataProyect/main/SQLScripts/athena"
OUT_DIR = Path("punto8") if Path("punto8").is_dir() else Path(".")

# Use local/EC2 credentials if present, otherwise prompt for AWS Academy Colab session keys
session = boto3.Session(region_name=REGION)
if session.get_credentials() is None:
    session = boto3.Session(
        region_name=REGION,
        aws_access_key_id=getpass.getpass("AWS_ACCESS_KEY_ID: ").strip(),
        aws_secret_access_key=getpass.getpass("AWS_SECRET_ACCESS_KEY: ").strip(),
        aws_session_token=getpass.getpass("AWS_SESSION_TOKEN: ").strip(),
    )

athena = session.client("athena")
s3 = session.client("s3")


def load_sql(filename: str) -> str:
    for candidate in (Path("SQLScripts/athena") / filename, Path("../SQLScripts/athena") / filename):
        if candidate.exists():
            return candidate.read_text(encoding="utf-8")
    with urllib.request.urlopen(f"{RAW_SQL_BASE}/{filename}", timeout=20) as resp:
        return resp.read().decode("utf-8")


def run_athena_query(sql_filename: str) -> pd.DataFrame:
    qid = athena.start_query_execution(
        QueryString=load_sql(sql_filename),
        QueryExecutionContext={"Database": "stocks_db"},
        ResultConfiguration={"OutputLocation": f"s3://{BUCKET}/athena-results/punto8/"},
    )["QueryExecutionId"]

    while True:
        status = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]
        state = status["State"]
        if state == "SUCCEEDED":
            break
        if state in ("FAILED", "CANCELLED"):
            raise RuntimeError(f"{sql_filename} {state}: {status.get('StateChangeReason')}")
        time.sleep(1.5)

    out_uri = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["ResultConfiguration"]["OutputLocation"]
    bucket, key = out_uri.replace("s3://", "", 1).split("/", 1)
    raw_csv = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
    df = pd.read_csv(io.BytesIO(raw_csv))
    assert not df.empty, f"Athena returned 0 rows for {sql_filename}"
    return df

In [ ]:
# Q1: Top 15 Tickers by Total Return %
q1 = run_athena_query("q1_top_tickers_by_return.sql").sort_values("total_return_pct", ascending=True)
display(q1.tail(5))

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(q1["ticker"], q1["total_return_pct"], color="#1f77b4")
ax.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=8)
ax.set_title("Q1 — Top 15 S&P 500 Tickers by Total Return % (Athena)")
ax.set_xlabel("Total Return (%)")
ax.grid(axis="x", linestyle="--", alpha=0.4)
plt.tight_layout()
fig.savefig(OUT_DIR / "chart_q1_top_gainers.png", dpi=150)
plt.show()

In [ ]:
# Q2: Average Daily Trading Volume by Sector (Top 15)
q2 = run_athena_query("q2_avg_volume_by_sector.sql").nlargest(15, "avg_daily_volume").sort_values("avg_daily_volume", ascending=True)
display(q2)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(q2["sector"], q2["avg_daily_volume"] / 1e6, color="#2ca02c")
ax.bar_label(bars, fmt="%.2fM", padding=3, fontsize=8)
ax.set_title("Q2 — Top 15 Sectors by Average Daily Volume (Athena)")
ax.set_xlabel("Average Daily Volume (Millions of Shares)")
ax.grid(axis="x", linestyle="--", alpha=0.4)
plt.tight_layout()
fig.savefig(OUT_DIR / "chart_q2_sector_volume.png", dpi=150)
plt.show()

In [ ]:
# Q3: Sector Market-Cap Ranking (Top 15)
q3 = run_athena_query("q3_sector_market_cap_ranking.sql").nlargest(15, "sector_market_cap").sort_values("sector_market_cap", ascending=True)
display(q3)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(q3["sector"], q3["sector_market_cap"] / 1e12, color="#9467bd")
ax.bar_label(bars, fmt="$%.2fT", padding=3, fontsize=8)
ax.set_title("Q3 — Top 15 Sectors by Total Deduplicated Market Cap (Athena)")
ax.set_xlabel("Total Market Cap (Trillions USD)")
ax.grid(axis="x", linestyle="--", alpha=0.4)
plt.tight_layout()
fig.savefig(OUT_DIR / "chart_q3_sector_market_cap.png", dpi=150)
plt.show()

In [ ]:
# Q4: Average Daily Return % vs Daily Volatility by Sector
q4 = run_athena_query("q4_daily_return_and_volatility.sql")
display(q4.head(10))

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(q4["daily_volatility_pct"], q4["avg_daily_return_pct"], s=60, color="#d62728", alpha=0.75)
label_rows = pd.concat([q4.nlargest(5, "daily_volatility_pct"), q4.nlargest(5, "avg_daily_return_pct")]).drop_duplicates("sector")
for _, row in label_rows.iterrows():
    ax.annotate(row["sector"], (row["daily_volatility_pct"], row["avg_daily_return_pct"]), xytext=(5, 4), textcoords="offset points", fontsize=8)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Q4 — Sector Daily Return vs Volatility (Athena)")
ax.set_xlabel("Daily Volatility (% Std Dev)")
ax.set_ylabel("Average Daily Return (%)")
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
fig.savefig(OUT_DIR / "chart_q4_risk_return.png", dpi=150)
plt.show()

In [ ]:
# Q5: Top 20 Tickers by Latest Close % vs SMA-50
q5 = run_athena_query("q5_sma_50.sql").sort_values("pct_vs_sma50", ascending=True)
display(q5.tail(5))

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(q5["ticker"], q5["pct_vs_sma50"], color="#ff7f0e")
ax.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=8)
ax.set_title("Q5 — Top 20 Tickers: Latest Close % Above SMA-50 (Athena)")
ax.set_xlabel("% vs SMA-50")
ax.grid(axis="x", linestyle="--", alpha=0.4)
plt.tight_layout()
fig.savefig(OUT_DIR / "chart_q5_sma50.png", dpi=150)
plt.show()